In [1]:
# Kaggle import for dataset
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sartajbhuvaji/brain-tumor-classification-mri")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brain-tumor-classification-mri' dataset.
Path to dataset files: /kaggle/input/brain-tumor-classification-mri


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# All the imports we need
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [4]:
# Loading the data
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder(root="/kaggle/input/brain-tumor-classification-mri/Training", transform=transform)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Implementing the VGG
# https://medium.com/@siddheshb008/vgg-net-architecture-explained-71179310050f
class CNNArch(nn.Module):
    def __init__(self):
        super(CNNArch, self).__init__()

        # Block 1: 224x224 -> 112x112
        self.conv1_1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.conv1_2 = nn.Conv2d(8, 8, kernel_size=3, padding=1)

        # Block 2: 112x112 -> 56x56
        self.conv2_1 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(16, 16, kernel_size=3, padding=1)

        # Block 3: 56x56 -> 28x28
        self.conv3_1 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3_2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)

        # Block 4: 28x28 -> 14x14
        self.conv4_1 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv4_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)

        # Block 5: 14x14 -> 7x7
        self.conv5_1 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.conv5_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, 4)

    def forward(self, x):
        # Block 1
        x = torch.relu(self.conv1_1(x))
        x = torch.relu(self.conv1_2(x))
        x = self.pool(x)

        # Block 2
        x = torch.relu(self.conv2_1(x))
        x = torch.relu(self.conv2_2(x))
        x = self.pool(x)

        # Block 3
        x = torch.relu(self.conv3_1(x))
        x = torch.relu(self.conv3_2(x))
        x = self.pool(x)

        # Block 4
        x = torch.relu(self.conv4_1(x))
        x = torch.relu(self.conv4_2(x))
        x = self.pool(x)

        # Block 5
        x = torch.relu(self.conv5_1(x))
        x = torch.relu(self.conv5_2(x))
        x = self.pool(x)

        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [ ]:
'''
Testing Model shapes

model = CNNArch().to(device)

dummy_input = torch.randn(1, 3, 224, 224).to(device)

output = model(dummy_input)

print(f"Output shape: {output.shape}") # Should be [1, 4]
'''

Output shape: torch.Size([1, 4])


In [ ]:
model = CNNArch().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 10
for epoch in range(epochs):
    running_loss = 0.0
    for i, (images, labels) in enumerate(dataloader):
      images = images.to(device)
      labels = labels.to(device)

      outputs = model(images)
      loss = criterion(outputs, labels)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
      if i % 10 == 9: # Print every 10 batches
          print(f'Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(dataloader)}], Loss: {running_loss/10:.4f}')
          running_loss = 0.0




Epoch [1/10], Step [10/90], Loss: 1.3736
Epoch [1/10], Step [20/90], Loss: 1.3700
Epoch [1/10], Step [30/90], Loss: 1.3401
Epoch [1/10], Step [40/90], Loss: 1.3565
Epoch [1/10], Step [50/90], Loss: 1.3501
Epoch [1/10], Step [60/90], Loss: 1.3459
Epoch [1/10], Step [70/90], Loss: 1.3333
Epoch [1/10], Step [80/90], Loss: 1.3433
Epoch [1/10], Step [90/90], Loss: 1.2878
Epoch [2/10], Step [10/90], Loss: 1.2258
Epoch [2/10], Step [20/90], Loss: 1.2653
Epoch [2/10], Step [30/90], Loss: 1.2163
Epoch [2/10], Step [40/90], Loss: 1.1564
Epoch [2/10], Step [50/90], Loss: 1.2209
Epoch [2/10], Step [60/90], Loss: 1.1844
Epoch [2/10], Step [70/90], Loss: 1.1543
Epoch [2/10], Step [80/90], Loss: 1.0535
Epoch [2/10], Step [90/90], Loss: 0.9851
Epoch [3/10], Step [10/90], Loss: 0.9754
Epoch [3/10], Step [20/90], Loss: 1.0539
Epoch [3/10], Step [30/90], Loss: 0.9985
Epoch [3/10], Step [40/90], Loss: 0.9160
Epoch [3/10], Step [50/90], Loss: 0.8967
Epoch [3/10], Step [60/90], Loss: 0.8540
Epoch [3/10], St

In [ ]:
# Evaluation Script
model.eval()
with torch.no_grad():
    test_images, test_labels = next(iter(dataloader))
    test_images = test_images.to(device)

    outputs = model(test_images)

    _, predicted = torch.max(outputs, 1)

print(f"Predicted: {predicted}")
print(f"Actual:    {test_labels}")
right = 0
for x in range(len(predicted)):
  if predicted[x] == test_labels[x]:
    right += 1
print(f"The amount of right ones are: {right}")

Predicted: tensor([3, 1, 1, 2, 0, 2, 3, 1, 3, 1, 2, 1, 0, 3, 1, 1, 2, 0, 1, 3, 0, 1, 2, 1,
        3, 2, 0, 3, 1, 3, 1, 3], device='cuda:0')
Actual:    tensor([2, 1, 0, 2, 1, 2, 3, 2, 3, 0, 3, 0, 0, 3, 2, 1, 2, 0, 1, 3, 0, 1, 2, 1,
        3, 2, 1, 3, 0, 1, 1, 3])
The amount of right ones are: 21
